# 結果を地図に落とす — 自治体が使える形にする（W8）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astrocamp-2026-siaPpts/astrocamp-sat-gis/blob/main/public/notebooks/tutorials/05_result_map.ipynb)

> **Private リポジトリの注意:** 上のバッジから直接開けない場合は
> [Google Colab](https://colab.research.google.com/) で「ファイル → ノートブックを開く → GitHub → プライベート リポジトリを含める」から開いてください。

## 本ノートで何をするか

W6 で作った **`field_features.csv`** と W7 で学んだ**判別・確信度**を、自治体が実際に使える
**成果物**にします。

- 判別マップ（**緑=作付 / 赤=休耕疑い / 灰=要確認**）
- **確信度（confidence）** を色で表すマップ
- 「**要確認優先リスト**」（現地見回りをどこから回るか）
- GIS で読める形式（`.gpkg` / `.shp` / `.geojson` / CSV）で出力
- **再現性**のためのヘッダ設定ブロック

> **自治体向けストーリー:** 衛星は「全部を判定する」のではなく、
> **「見に行くべき場所」を絞る**のに使います。確信度が高い圃場は自動判定で済ませ、
> グレーゾーンだけを現地確認すれば、調査コストが大きく下がります（W7 の費用対効果）。


---

## 学習目標
- 判別結果を **field_id で圃場ポリゴンに結合**して地図にする
- **確信度（`predict_proba`）** で「要確認」を抽出し、**現地確認優先リスト**を作る
- 結果を **CSV / Shapefile / GeoPackage** に出力し、QGIS などで開けるようにする
- **再現性**（誰が実行しても同じ結果）のための設定ブロックを書ける


In [ ]:
# 必要なライブラリのインストール（Colab / ローカルの Jupyter どちらでも同じ）
!pip install -q geopandas shapely matplotlib japanize-matplotlib scikit-learn pandas numpy


In [ ]:
# ── ライブラリ読み込み ──
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

try:
    import japanize_matplotlib  # あれば使う（無くても env.setup_font() が日本語対応）
except ImportError:
    japanize_matplotlib = None  # 未インストールでも続行（ローカルはシステムフォントで表示）

# ---- 共通環境ヘルパー（Colab / ローカル両対応） ----
try:
    from env import setup_font
    setup_font()
except ImportError:
    print('（env.py が無いため、日本語フォント設定をスキップ）')
print('OK')


## 再現性ヘッダー — 設定を1か所にまとめる

03・04 とまったく同じ値を使う（`resources/product-related/guide_camp.md` §5 が全教材共通の設定）。


In [ ]:
# ============================================================
# 05: 解析結果を地図・GIS 成果物にする
# 全教材で共通の設定（guide_camp.md §5 と同じ値）
# ============================================================
TARGET_YEAR = 2024          # 判別・地図化の対象年
CRS         = 'EPSG:32654'  # 集計に使った座標系
SCALE       = 10            # m

VH_FLOOD_THRESH  = -18.0    # dB  5月の湛水判定（vh_may）
NDVI_HIGH_THRESH = 0.70     #     8月の作付判定（ndvi_aug）
NDVI_LOW_THRESH  = 0.25     #     年間最大が低い＝休耕（ndvi_max）

GRAY_LOW, GRAY_HIGH = 0.35, 0.65   # グレーゾーン（要確認）の確信度レンジ
print(f'対象年 {TARGET_YEAR} / {CRS} / scale={SCALE}m')


## データを読み込む（W6 の成果物）

- `field_features.csv` : 「1筆=1行」の特徴量表（04 で作成）
- `fields.geojson`     : 圃場ポリゴンのジオメトリ（04 で作成）

まだ無い場合は、動作確認用の**合成データ**にフォールバックします。


In [ ]:
# ── W6 の成果物を読み込む（無ければ合成データでフォールバック） ──
import os
import json
from shapely import geometry

FEATURES = ['ndvi_max', 'ndvi_amplitude', 'ndvi_may', 'ndvi_aug', 'vh_may', 'vh_aug',
            'vh_min', 'vh_amplitude', 'vh_temporal_diff', 'vv_vh_may', 'vv_vh_aug',
            'growth_slope', 'ndvi_max_month', 'ndvi_min']

def make_fallback():
    rng = np.random.default_rng(7)
    n = 60
    geoms = []
    for i in range(n):
        x = rng.uniform(140.93, 141.04); y = rng.uniform(37.60, 37.82)
        dx, dy = rng.uniform(0.004, 0.008), rng.uniform(0.004, 0.008)
        geoms.append(geometry.box(x, y, x + dx, y + dy))
    gdf = gpd.GeoDataFrame({'field_id': [f'field_{i:03d}' for i in range(n)]},
                           geometry=geoms, crs='EPSG:4326')
    types = rng.choice(['作付', '休耕', '森林'], size=n, p=[0.5, 0.3, 0.2])
    base = {
        '作付': dict(ndvi_max=0.78, ndvi_amplitude=0.50, ndvi_may=0.45, ndvi_aug=0.72,
                     vh_may=-19.0, vh_aug=-12.0, vh_min=-19.5, vh_amplitude=7.0,
                     vh_temporal_diff=7.0, vv_vh_may=0.50, vv_vh_aug=0.55,
                     growth_slope=0.12, ndvi_max_month=8, ndvi_min=0.25),
        '休耕': dict(ndvi_max=0.26, ndvi_amplitude=0.05, ndvi_may=0.21, ndvi_aug=0.24,
                     vh_may=-13.0, vh_aug=-13.0, vh_min=-13.5, vh_amplitude=1.0,
                     vh_temporal_diff=0.2, vv_vh_may=0.58, vv_vh_aug=0.58,
                     growth_slope=0.01, ndvi_max_month=7, ndvi_min=0.19),
        '森林': dict(ndvi_max=0.84, ndvi_amplitude=0.06, ndvi_may=0.80, ndvi_aug=0.82,
                     vh_may=-9.0, vh_aug=-9.0, vh_min=-9.5, vh_amplitude=0.8,
                     vh_temporal_diff=0.1, vv_vh_may=0.60, vv_vh_aug=0.60,
                     growth_slope=0.01, ndvi_max_month=8, ndvi_min=0.78),
    }
    sd = {'作付': 0.03, '休耕': 0.02, '森林': 0.02}
    rows = []
    for fid, t in zip(gdf['field_id'], types):
        for y in [2023, 2024, 2025]:
            row = {'field_id': fid, 'year': y}
            for k in FEATURES:
                row[k] = float(rng.normal(base[t][k], sd[t]))
            rows.append(row)
    return gdf, pd.DataFrame(rows)

SOURCE = 'W6 の成果物（field_features.csv + fields.geojson）'
# ファイルはカレント → tutorials/data の順で探す（Colab / ローカル両対応）
try:
    from env import data_dir
    _SEARCH = ['', data_dir()]
except ImportError:
    _SEARCH = ['']
def _find(name):
    for c in _SEARCH:
        p = os.path.join(c, name) if c else name
        if os.path.exists(p):
            return p
    return name
_FCSV, _FGEO = _find('field_features.csv'), _find('fields.geojson')
try:
    assert os.path.exists(_FCSV) and os.path.exists(_FGEO)
    field_features = pd.read_csv(_FCSV)
    with open(_FGEO) as f:
        fc = json.load(f)
    fields_gdf = gpd.GeoDataFrame.from_features(fc['features'], crs='EPSG:4326')
    # 念のため field_id を文字列で揃える
    fields_gdf['field_id'] = fields_gdf['field_id'].astype(str)
except Exception as e:
    print(f'[フォールバック] W6 成果物が無いため合成データで続行します: {e}')
    fields_gdf, field_features = make_fallback()
    SOURCE = '合成データ（フォールバック）'

print('データソース:', SOURCE)
print('特徴量表:', field_features.shape, '/ 圃場ポリゴン:', len(fields_gdf))
field_features.head()


## 判別を適用する（物理ルール → 疑似ラベル → RandomForest → 確信度）

03・W7 の流れをそのまま使います。

1. 物理ルールで**疑似ラベル**を作る（`vh_may ≤ -18dB かつ ndvi_aug ≥ 0.70` = 作付 ／ `ndvi_max < 0.25` = 休耕）
2. **RandomForest** で学習・予測
3. `predict_proba` で「作付である確率」= **確信度**を取り出す


In [ ]:
# ── 判別と確信度の計算 ──
from sklearn.ensemble import RandomForestClassifier

res = field_features.copy()

# ① 物理ルールで疑似ラベル（03 と同じ閾値）
res['rule_label'] = np.nan
res.loc[(res['vh_may'] <= VH_FLOOD_THRESH) & (res['ndvi_aug'] >= NDVI_HIGH_THRESH),
        'rule_label'] = 1                                                      # 作付
res.loc[res['ndvi_max'] < NDVI_LOW_THRESH, 'rule_label'] = 0                   # 休耕
labeled = res[res['rule_label'].notna()].copy()
labeled['rule_label'] = labeled['rule_label'].astype(int)
print('疑似ラベル付きサンプル:', len(labeled), '/', len(res))

# ② RandomForest（W7: 時空間リークを避けるため field_id 単位で評価用を分ける）
from sklearn.model_selection import train_test_split
test_fids = pd.Series(labeled['field_id'].unique()).sample(frac=0.33, random_state=42)
tr = labeled[~labeled['field_id'].isin(test_fids)]
te = labeled[labeled['field_id'].isin(test_fids)]

assert len(tr) > 0 and tr['rule_label'].nunique() >= 2, (
    '学習データが足りない（疑似ラベルが片方のクラスしかない）。'
    'Step1 に戻って閾値を調整すること。')

model = RandomForestClassifier(n_estimators=200, max_depth=4,
                               random_state=42, class_weight='balanced')
model.fit(tr[FEATURES], tr['rule_label'])
from sklearn.metrics import accuracy_score
print('field_id 分割の精度:', round(accuracy_score(te['rule_label'],
                                     model.predict(te[FEATURES])), 3))

# ③ 全圃場を予測し、「作付である確信度」を取り出す
res['pred'] = model.predict(res[FEATURES])
res['prob_crop'] = model.predict_proba(res[FEATURES])[:, 1]
res['confidence'] = np.maximum(res['prob_crop'], 1 - res['prob_crop'])
res['zone'] = np.where(res['prob_crop'] >= GRAY_HIGH, '作付（確信）',
              np.where(res['prob_crop'] <= GRAY_LOW, '休耕（確信）', 'グレーゾーン（要確認）'))
print('確信度ゾーンの内訳:')
print(res['zone'].value_counts())


## 判別マップを作る（緑=作付 / 赤=休耕疑い / 灰=要確認）

最新年（データがあれば2025年）の判別結果を **field_id で圃場ポリゴンに結合**して地図にします。


In [ ]:
# ── field_id で結合して判別マップを作る ──
# 対象年は再現性ヘッダーの TARGET_YEAR を使う。
# res['year'].max() にすると取得期間の最終年（2025）になり、
# 教材が指定している解析対象年（2024）とずれる。
latest_year = TARGET_YEAR if (res['year'] == TARGET_YEAR).any() else res['year'].max()
if latest_year != TARGET_YEAR:
    print(f'[注意] {TARGET_YEAR} 年のデータが無いため {latest_year} 年で作図します')
res_latest = res[res['year'] == latest_year].copy()
gdf = fields_gdf.merge(res_latest, on='field_id', how='inner')
print('結合できた筆数:', len(gdf), '/ 最新年:', latest_year)

zmap = {'作付（確信）': '#2e7d32', '休耕（確信）': '#d32f2f', 'グレーゾーン（要確認）': '#9e9e9e'}
fig, ax = plt.subplots(figsize=(10, 6))
for z, c in zmap.items():
    sub = gdf[gdf['zone'] == z]
    if len(sub):
        sub.plot(ax=ax, color=c, edgecolor='white', linewidth=0.4, label=z)
ax.set_title(f'作付／休耕の判別マップ（{latest_year}）— 緑=作付 赤=休耕疑い 灰=要確認')
ax.legend(loc='lower right', fontsize=10)
ax.set_axis_off()
plt.tight_layout(); plt.show()


## 確信度マップと「要確認優先リスト」

判別結果に**確信度**を併記すると、自治体職員が「どこを現地確認するか」を優先度付きで決められます。
**確信度が低い圃場（グレーゾーン）ほど、優先して現地確認**するリストを作ります。


In [ ]:
# ── 確信度マップ ──
fig, ax = plt.subplots(figsize=(10, 5))
gdf.plot(ax=ax, column='confidence', cmap='RdYlGn', legend=True, edgecolor='white', linewidth=0.4)
ax.set_title(f'確信度マップ（{latest_year}）— 緑=確信度高（自動判定でOK） 赤=確信度低（要確認）')
ax.set_axis_off()
plt.tight_layout(); plt.show()

# ── 要確認優先リスト（確信度が低い順） ──
gray = res[res['zone'] == 'グレーゾーン（要確認）'].sort_values('confidence')
print('要確認（グレーゾーン）圃場数:', len(gray), '/ 全体:', len(res))
print(gray[['field_id', 'year', 'prob_crop', 'confidence']].head(10))
gray.to_csv('priority_check_list.csv', index=False)
print('保存: priority_check_list.csv（現地確認優先リスト）')


## GIS で読める形に出力する（Shapefile / GeoPackage / CSV）

QGIS などの GIS や e-Stat データと重ねられるよう、**圃場ポリゴンに結果を結合した状態**で保存します。
GeoPackage（.gpkg）は1ファイルに全部入るのでおすすめです。


In [ ]:
# ── GIS 出力 ──
out_cols = ['field_id', 'year', 'rule_label', 'pred', 'prob_crop', 'confidence', 'zone', 'geometry']
out = gdf[out_cols].copy()

# GeoPackage を主とする。Shapefile は列名10文字までの制限があり、
# 日本語の属性値は encoding を明示しないと文字化けする（互換のため従として出力）。
out.to_file('fukushima_field_results.gpkg', driver='GPKG')          # GeoPackage（推奨）
out.to_file('fukushima_field_results.shp', encoding='utf-8')        # Shapefile（従）
out.to_file('fukushima_field_results.geojson', driver='GeoJSON')    # GeoJSON
out.drop(columns='geometry').to_csv('fukushima_field_results.csv', index=False)

print('出力一覧:')
for f in ['fukushima_field_results.gpkg', 'fukushima_field_results.shp',
          'fukushima_field_results.geojson', 'fukushima_field_results.csv']:
    print(' -', f, os.path.getsize(f) if os.path.exists(f) else '')


## 再現性 — 「誰が実行しても同じ結果になる」ための設定ブロック

解析の前提（期間・範囲・閾値・使用データ）をコードの冒頭に1か所にまとめると、
**再現性**（誰が実行しても同じ結果）が確保できます。合宿ではノートを
`01_download → 02_preprocess → 03_feature_engineering → 04_classification → 05_validation`
のように**番号付きで並べる**ことを推奨します（規約として README にも記載）。


In [ ]:
# ── 再現性ヘッダ設定ブロック（解析の「注文書」） ──
# 値は 03・04 と共通（guide_camp.md §5）。変えたらここも必ず書き換える。
CONFIG = {
    'acquisition_period': f'{"2021-01-01"} .. {"2025-12-31"}（取得期間・1筆×1年=1行）',
    'target_year': TARGET_YEAR,
    'season_window': [3, 4, 5, 6, 7, 8, 9, 10],
    'aoi': '[140.90, 37.60, 141.04, 37.83]（相馬市 + 南相馬市東側）',
    'crs': CRS,
    'scale_m': SCALE,
    'sentinel2': {'collection': 'COPERNICUS/S2_SR_HARMONIZED',
                  'scene_cloud_threshold': 20,
                  'pixel_cloud_mask': 'SCL（雲9・影3・cirrus10・cloud8 を除去）',
                  'bands': ['B8', 'B4'],
                  'ndvi': 'normalizedDifference(B8, B4)'},
    'sentinel1': {'collection': 'COPERNICUS/S1_GRD',
                  'mode': 'IW',
                  'orbit': 'DESCENDING',
                  'polarization': ['VV', 'VH'],
                  'unit': 'dB（S1_GRDは既にdB。二重変換禁止）',
                  'vv_vh': 'VV_db - VH_db（dBでは比＝差。割り算は不可）'},
    'rule_thresholds': {'vh_may <=': VH_FLOOD_THRESH,
                        'ndvi_aug >=': NDVI_HIGH_THRESH,
                        'ndvi_max <': NDVI_LOW_THRESH},
    'gray_zone': f'{GRAY_LOW} < prob_crop < {GRAY_HIGH}',
    'parcel_polygon': '農水省筆ポリゴン（fukushima_polygons.geojson）',
    'validation_data': 'validation_labels.csv（目視 20〜30 筆）',
    'random_seed': 42,
    'outputs': ['field_features.csv', 'fields.geojson',
                'fukushima_field_results.gpkg', 'priority_check_list.csv'],
}
for k, v in CONFIG.items():
    print(f'{k}: {v}')


## 自治体向けの説明ストーリー（発表の型）

この成果物を自治体に説明するときの骨格です。

> **衛星は「全部を判定する」のではなく、「見に行くべき場所」を絞ります。**

1. 確信度が高い圃場（緑=作付 / 赤=休耕）は**自動判定**で概ね分類できる
2. グレーゾーン（灰）は**現地確認の優先対象**——確信度が低い順にリスト化した
3. 現地調査を全圃場に回るより、**要確認（15〜20%）に絞る**ことでコストを削減できる（W7 の費用対効果）
4. 圃場単位の結果は **GeoPackage / Shapefile** で配布でき、既存の農地台帳・GIS と重ねられる


In [ ]:
# ── 成果サマリー（発表で使う数字） ──
print('対象筆数:', len(gdf))
print()
print('判定内訳（最新年）:')
print(gdf['zone'].value_counts().rename('筆数'))
n_gray = int((gdf['zone'] == 'グレーゾーン（要確認）').sum())
print()
print(f'→ 現地確認対象（グレーゾーン）: {n_gray} 筆 / {len(gdf)} 筆（{n_gray / len(gdf) * 100:.0f}%）')
print('  自動判定で絞り、人間は「見に行くべき場所」だけを確認すればよい。')


## 費用対効果を見積もる — 「何時間減るか」で語る

判別マップだけでは自治体は動けない。**「現地確認が何時間減るか」**に翻訳して初めて提案になる。

```text
削減効果 = N × t × (1 − r)
  N … 対象圃場数（筆）
  t … 1筆あたりの現地確認時間（分）
  r … 要確認率（グレーゾーンの割合）
```

`r` は解析の精度で決まる。確信度の高い圃場を自動判定に任せられるほど `r` は下がり、削減効果が上がる。


In [ ]:
# ── 費用対効果の試算（自治体向けスライドの数字はここから作る） ──
N_TOTAL   = len(gdf)          # 対象圃場数（筆）
T_FIELD   = 10                # 1筆あたりの現地確認時間（分）※自治体ヒアリングの想定値
r         = n_gray / N_TOTAL  # 要確認率 = グレーゾーンの割合

before_min = N_TOTAL * T_FIELD              # 従来: 全筆を現地確認
after_min  = N_TOTAL * T_FIELD * r          # 提案後: グレーゾーンだけ確認
saved_min  = before_min - after_min

print(f'対象圃場数 N = {N_TOTAL} 筆 / 1筆あたり t = {T_FIELD} 分')
print(f'要確認率  r = {r:.2f}（グレーゾーン {n_gray} 筆）')
print()
print(f'従来（全筆確認）  : {before_min:,} 分 = {before_min/60:.1f} 時間')
print(f'提案後（絞り込み）: {after_min:,.0f} 分 = {after_min/60:.1f} 時間')
print(f'削減効果          : {saved_min:,.0f} 分 = {saved_min/60:.1f} 時間'
      f'（{(1-r)*100:.0f}% 削減）')
print()

# 要確認率を振ったときの感度（スライドには「rは精度次第で変わる」と添える）
print('要確認率 r を変えたときの年間確認時間:')
for rr in [0.10, 0.20, 0.30, 0.50]:
    h = N_TOTAL * T_FIELD * rr / 60
    print(f'  r = {rr:.2f} → {h:5.1f} 時間（{(1-rr)*100:.0f}% 削減）')
print()
print('※ t は自治体の実務時間、N は対象エリアの実筆数に置き換えること。')
print('  r は自チームの解析結果から出た値であり、精度が上がれば下がる。')
